# Modeling Validation

Validates trend calculations, clustering stability across random seeds, coordinate accuracy, and checks for data leakage.

In [ ]:
import sys, logging, warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from src.loaders import GreenSentinelLoader, DKVLoader
from src.processors.feature_engineer import FeatureEngineer
from src.models.gentrification_model import GentrificationRiskModel

## 1. Manual trend check

Verify a turbine-free linear regression on one station against the feature stack.

In [ ]:
gs = GreenSentinelLoader().run().get_data()
fe = FeatureEngineer(gs).create_all_features()

from scipy import stats
st = gs[(gs['station']=='DEB-KER11') & (gs['measurement_type']=='PM2.5')]
daily = st.groupby(st['timestamp'].dt.date)['value'].mean().values
slope, *_ = stats.linregress(np.arange(len(daily)), daily)
print(f'Manual slope DEB-KER11 PM2.5: {slope:.4f}')
print(f'Feature trend from pipeline : {fe[fe.station=="DEB-KER11"]["PM2.5_trend"].values[0]:.4f}')

## 2. Clustering stability

Run K-means with multiple seeds and compare cluster assignments with the Adjusted Rand Index.

In [ ]:
cols = ['env_improvement_index','current_PM2.5','pm25_volatility']
X = StandardScaler().fit_transform(fe[cols].fillna(0).values)

ref = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)
scores = []
for seed in range(20):
    pred = KMeans(n_clusters=4, random_state=seed, n_init=10).fit_predict(X)
    scores.append(adjusted_rand_score(ref, pred))

print(f'Adjusted Rand Index across 20 seeds: mean={np.mean(scores):.3f} min={np.min(scores):.3f}')
print('Silhouette (42 seed):', round(silhouette_score(X, ref), 3))

## 3. Coordinate sanity

All stations should be within a few kilometers of Debrecen (47.53, 21.63).

In [ ]:
coords = gs[['station','latitude','longitude']].drop_duplicates()
print(coords.to_string(index=False))
assert coords['latitude'].between(47.0, 48.0).all()
assert coords['longitude'].between(21.0, 22.0).all()
print('\nAll coordinates within Debrecen bounds.')

## 4. No data leakage

The current-pollution window (last 7 days) and 30-day trend use the full series — no train/test split is applied because the model is unsupervised and descriptive. Save final risk report.

In [ ]:
dkv = DKVLoader().run()
transit = dkv.compute_transit_accessibility(gs[['station','latitude','longitude']].drop_duplicates())
fe = FeatureEngineer(gs, transit_data=transit)
features = fe.create_all_features()
model = GentrificationRiskModel(features).fit_clustering()
report = model.generate_risk_report()
model.save()
print(report[['station','risk_category','env_improvement_index','current_PM2.5']].to_string(index=False))
print('\nMetrics saved to output/models/model_metrics.json')